http://rosalind.info/problems/lcsm/

*Given*: A collection of k (k≤100) DNA strings of length at most 1 kbp each in FASTA format.

_Return_: A longest common substring of the collection. (If multiple solutions exist, you may return any single solution.)

In [34]:
from Bio.Seq import Seq
from Bio import SeqIO

def read_fasta(fasta_file):
    '''Read fasta file and return the list of records, consisting of
    description and sequence'''
    handle = open(fasta_file)
    records = list(SeqIO.parse(handle, "fasta"))
    handle.close()
    return [(str(rec.seq)) for rec in records]

In [29]:
def longest_common_substring(s1, s2):
   m = [[0] * (1 + len(s2)) for i in range(1 + len(s1))]
   longest, x_longest = 0, 0
   for x in range(1, 1 + len(s1)):
       for y in range(1, 1 + len(s2)):
           if s1[x - 1] == s2[y - 1]:
               m[x][y] = m[x - 1][y - 1] + 1
               if m[x][y] > longest:
                   longest = m[x][y]
                   x_longest = x
           else:
               m[x][y] = 0
   return s1[x_longest - longest: x_longest]

In [37]:
def calc_cache_pos(strings, indexes):
    factor = 1
    pos = 0
    for s, i in zip(strings, indexes):
        pos += i * factor
        factor *= len(s)
    return pos

def lcs_back(strings, indexes, cache):
    if -1 in indexes:
        return ""
    match = all(strings[0][indexes[0]] == s[i]
                for s, i in zip(strings, indexes))
    if match:
        new_indexes = [i - 1 for i in indexes]
        result = lcs_back(strings, new_indexes, cache) + strings[0][indexes[0]]
    else:
        substrings = [""] * len(strings)
        for n in range(len(strings)):
            if indexes[n] > 0:
                new_indexes = indexes[:]
                new_indexes[n] -= 1
                cache_pos = calc_cache_pos(strings, new_indexes)
                if cache[cache_pos] is None:
                    substrings[n] = lcs_back(strings, new_indexes, cache)
                else:
                    substrings[n] = cache[cache_pos]
        result = max(substrings, key=len)
    cache[calc_cache_pos(strings, indexes)] = result
    return result

def lcs(strings):
    """
    >>> lcs(['666222054263314443712', '5432127413542377777', '6664664565464057425'])
    '54442'
    >>> lcs(['abacbdab', 'bdcaba', 'cbacaa'])
    'baa'
    """
    if len(strings) == 0:
        return ""
    elif len(strings) == 1:
        return strings[0]
    else:
        cache_size = 1
        for s in strings:
            cache_size *= len(s)
        cache = [None] * cache_size
        indexes = [len(s) - 1 for s in strings]
        return lcs_back(strings, indexes, cache)

In [39]:
import itertools
dataset = read_fasta("datasets/rosalind_lcsm_example.txt")

print (dataset)

['GATTACA', 'TAGACCA', 'ATACA']


In [40]:
print(lcs(dataset))

AACA


In [44]:
dataset = read_fasta("datasets/rosalind_lcsm.txt")

def long_substr(data):
    substr = ''
    if len(data) > 1 and len(data[0]) > 0:
        for i in range(len(data[0])):
            for j in range(len(data[0])-i+1):
                if j > len(substr) and is_substr(data[0][i:i+j], data):
                    substr = data[0][i:i+j]
    return substr

def is_substr(find, data):
    if len(data) < 1 and len(find) < 1:
        return False
    for i in range(len(data)):
        if find not in data[i]:
            return False
    return True


print (long_substr(dataset))

ATAAAACGAAAGAACCCCGTGGTGGCAACGCAGTGTGGTCTCAGACTCTAGGGGTTTCGCATCCTCGAATAGCCATTCATCCAAATCCACGACCTAGCCGCCACAGAAGATACACGTCGTGAATGCGAACATGTAGACGCCGCGGAGTACACGAGGAATAAGACCCTGTAATAGCCTCATCATTGCGGGCGACGCAAAGCAGTCCTCCTCTCTGGTCGGTCCCGATCTGAGGGCATGTCCTCAACTATGCGGGATGTGAGGATC
